In [ ]:
import matplotlib.pylab as plt 
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge

# 1. Cost functions

Lets focus on linear regression of the form 

$\mathbf{y} \approx f(\mathbf{X}) = \mathbf{X}\mathbf{w_1} + \mathbf{w_0}.$


#### 1.1 What are the rows of $\mathbf{X}$?

__< Your answer >__

#### 1.2 What are the columns of $\mathbf{X}$?

__< Your answer >__

Often, we write the equation above as

$\mathbf{y} \approx \mathbf{\tilde{X}}\mathbf{w}$

#### 1.3 How does $\mathbf{\tilde{X}}$ look like in this case (i.e., how does the shape of the matrix change compared to $\mathbf{X}$)?

__< Your answer >__

For machine learning, we need a cost function. Two common choices are the mean-squared error (MSE, $\mathcal{L}_2$), and the mean-absolute error (MAE, $\mathcal{L}_1$)

\begin{align}
    \mathcal{L}_2 &=& \frac{1}{N} \sum_{i=1}^N \left(y_i - f(x_i) \right)^2 \\
    \mathcal{L}_1 &=& \frac{1}{N} \sum_{i=1}^N \left|y_i - f(x_i) \right| 
\end{align}

#### 1.4 In the Jupyter notebook, write a Python function that computes these two cost functions given an error term $\boldsymbol{\epsilon} = \mathbf{y} - \mathbf{\tilde{X}}\mathbf{w}$

In [ ]:
def mean_squared_error(error_vector):
    mean_squared_error = ... # fill your code
    return mean_squared_error

In [ ]:
def mean_absolute_error(error_vector):
    mean_absolute_error = ... # fill your code
    return mean_absolute_error

Your code should run as follows

```python
mean_squared_error(np.array([0,0,0]))
> returns 0
```

```python
mean_squared_error(np.array([1,1,1]))
> returns 1
```

#### 1.5 What is the shape of these cost functions as a function of the error

In [ ]:
x_axis = np.linspace(-10,10,100) # change as you wish for your plot
y_mae = [mean_absolute_error(x) for x in x_axis]
y_mse = [mean_squared_error(x) for x in x_axis]

In [ ]:
plt.plot(x_axis, y_mse, label='MSE')
plt.plot(x_axis, y_mae, label='MAE')
plt.xlabel('error term')
plt.ylabel('cost function')

#### 1.6  Are both loss functions differentiable for all $\boldsymbol{\epsilon}$? What implications does this have for gradient based optimization like gradient descent?

__< Your answer >__

#### 1. Which loss function is more sensitive to outliers and why?

__< Your answer >__

# 2. Regularization

Assume that the columns of $\mathbf{X}$ are linearly independent.
As a refresher of linear algebra, recall when the linear system $\mathbf{X}\mathbf{w} = \mathbf{y}$ has

#### 2.1 One unique solution

Let $\mathbf{X}$ be an $N \times p$ matrix ($N$ samples, $p$ unknowns) and $[\mathbf{X}|\mathbf{y}]$ the augmented matrix. The system has a unique solution when $\text{rank}(\mathbf{X}) = \text{rank}([\mathbf{X}|\mathbf{y}]) = p$, which is $\mathbf{y}$ lies in the column space of $\mathbf{X}$ and there are no free variables. Since the columns of $\mathbf{X}$ are linearly independent, $\text{rank}(\mathbf{X}) = p$ already holds (this needs $N \geq p$). If $\mathbf{X}$ is square ($N = p$), it is invertible and $\mathbf{w} = \mathbf{X}^{-1}\mathbf{y}$ for any $\mathbf{y}$.

#### 2.2 No solution

There is no solution when $\text{rank}(\mathbf{X}) < \text{rank}([\mathbf{X}|\mathbf{y}])$. Then $\mathbf{y}$ is not in the column space of $\mathbf{X}$, so no combination of the columns gives $\mathbf{y}$. This is what usually happens in regression: we have more samples than parameters ($N > p$) and the data is noisy, so instead of solving the system exactly we minimize the error.

#### 2.3 An infinite number of solutions

There are infinitely many solutions when $\text{rank}(\mathbf{X}) = \text{rank}([\mathbf{X}|\mathbf{y}]) < p$. In this case $\mathbf{X}$ has a non-trivial null space, so if $\mathbf{w}_0$ is a solution, $\mathbf{w}_0 + \mathbf{z}$ is also one for any $\mathbf{z}$ with $\mathbf{X}\mathbf{z} = \mathbf{0}$ (there are $p - \text{rank}(\mathbf{X})$ free parameters). This always happens when there are more columns than rows ($p > N$) and the system is consistent, because $\text{rank}(\mathbf{X}) \leq N < p$. In this case the columns can't be linearly independent.

#### 2.4 Give a geometrical interpretation of the matrix Rank (10 bonus point if you use the Manim package)

The rank is the maximum number of linearly independent columns of $\mathbf{X}$, i.e. the dimension of the space they span. If we look at $\mathbf{X}$ as a linear map $\mathbf{w} \mapsto \mathbf{X}\mathbf{w}$ from $\mathbb{R}^p$ to $\mathbb{R}^N$, the rank is the dimension of the set of all possible outputs $\mathbf{X}\mathbf{w}$: with rank 1 everything is mapped onto a line, with rank 2 onto a plane, and so on. If $\text{rank}(\mathbf{X}) < p$, some directions (the null space) are squashed to zero, so the map loses information and cannot be inverted (for a square matrix this means $\det \mathbf{X} = 0$). This is also why $\mathbf{X}\mathbf{w} = \mathbf{y}$ only has a solution if $\mathbf{y}$ lies in this subspace, and why the solution is not unique when some directions are lost.

**Bonus (Manim):** the animation below shows this for $2 \times 2$ matrices. A rank-2 matrix maps the grid onto the whole plane and can be undone. A rank-1 matrix (second column $= 2 \times$ first column) squashes the plane onto a line: the null space (yellow) is mapped to zero, and a point $\mathbf{y}$ that is not on the line cannot be reached. The code is in `rank_animation.py` (run `manim -qm rank_animation.py RankInterpretation`).

![Rank animation](rank_animation.gif)

#### 2.5 In general, why can't we solve the linear equation using $\mathbf{y} = \tilde{\mathbf{X}}^{-1}\textbf{w}$? (1 point)

There are two problems with this expression. First, it is not the right rearrangement: from $\mathbf{y} = \tilde{\mathbf{X}}\mathbf{w}$ we would get $\mathbf{w} = \tilde{\mathbf{X}}^{-1}\mathbf{y}$, not $\mathbf{y} = \tilde{\mathbf{X}}^{-1}\mathbf{w}$. Also, $\mathbf{y}$ is the data we already know and $\mathbf{w}$ is what we want to find, so this equation does not help us. Second, $\tilde{\mathbf{X}}$ is $N \times (p+1)$ and usually not square ($N \gg p+1$), so $\tilde{\mathbf{X}}^{-1}$ does not exist. Even a square $\tilde{\mathbf{X}}$ is singular if its columns are linearly dependent.

On top of that, with noisy data there is usually no exact solution (see 2.2), so we minimise $\|\mathbf{y} - \tilde{\mathbf{X}}\mathbf{w}\|_2^2$ instead.

#### 2.6 Differentiate above formula step by step and show what will we have?

__< Your answer >__

#### 2.7 if we want $\| y - \tilde{X} w \|_2^2$ to be minimum, what should the derivative be equal to?

__< Your answer >__

#### 2.8 What is the Hat matrix and what does its diagonal values correspond to?

__< Your answer >__

#### 2.9 What is Willson’s plot and how does it help in outlier detection?

__< Your answer >__

#### 2.10 What happens if some columns are linearly dependent? What is the connection to feature selection?

__< Your answer >__

#### 2.11 What will be the new cost function after adding the regularization term?

__< Your answer >__

#### 2.12 What will be the new 𝑤 when we differentiate the new cost function and set it to zero.

__< Your answer >__

#### 2.13 Prove that the part of Hat matrix where we want to take the inverse from is always reversible after we introduce the regularization term.

__< Your answer >__

#### 2.14  What is the shape of the parabola as a function of $a$?

In [ ]:
def parabola(x, a = 0.5): 
    return a * x ** 2

In [ ]:
x_axis_parabola = np.linspace(-10, 10, 100)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300) #just to have a nicer image
plt.plot(x_axis_parabola, parabola(x_axis_parabola), color=plt.cm.magma(0.25),) #because i love magma palette

Increasing $|a|$ makes the parabola narrower, and vice versa; with positve $a$, parabola points upwards, with negative - downwards; $a = 0$ gives a straight line. 

#### 2.15 Plot the approximation to the function for different order polynomials ($N \in \{1, 2, 16\}$) and with different regularization strength ($\lambda \in \{0, 10^{-3}, 10^{-2}, 1\}$). What do you observe 

In [ ]:
def true_function(X):
    return np.cos(1.5 * np.pi * X)

In [ ]:
X_test = np.linspace(0, 1, 100) # some grid for us on the x axis

In [ ]:
n_samples = 10 # the number of points we will sample from true_function
degrees = [1, 2, 16] # the polynomial degrees we will test

X = np.sort(np.random.rand(n_samples))
y = true_function(X) + np.random.randn(n_samples) * 0.1 # add some scaled random noise

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=300) #just to have a nicer image
plt.scatter(X, y,color=plt.cm.magma(0.55), label='noisy samples')
plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.25),  label='true function')
plt.legend()

The following code will fit a polynomial regression, you need to fill the degree

In [ ]:
polynomial_features = PolynomialFeatures(degree=8,
                                             include_bias=False)
linear_regression = LinearRegression()
pipeline = Pipeline([("polynomial_features", polynomial_features),
                     ("linear_regression", linear_regression)])
pipeline.fit(X[:, np.newaxis], y)

To plot the result, you can use the following code

In [ ]:
#plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="model of degree ")
#plt.scatter(X, y, label='noisy samples')
#plt.plot(X_test, true_function(X_test), c='r', label='true function')
#plt.legend()

In [ ]:
# let's combine it in one cell:

fig, ax = plt.subplots(figsize=(8, 5),dpi=300) #just to have a nice picture
colors = plt.cm.magma(np.linspace(0.35, 0.85, len(degrees))) #and of course we'll use our favourite magma 

for degree, color in zip(degrees,colors) :
    polynomial_features = PolynomialFeatures(degree=degree,
                                             include_bias=False)
    linear_regression = LinearRegression()
    pipeline = Pipeline([("polynomial_features", polynomial_features),
                        ("linear_regression", linear_regression)])
    pipeline.fit(X[:, np.newaxis], y)

    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]),
             color=color,
             label=f"model of degree {degree}")

plt.scatter(X, y, s=20, color=plt.cm.magma(0.15), label="noisy samples")
plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.15), label="true function")
plt.legend()
plt.show()
    

Here, we see that the model of degree 1 approximates the true function rather badly, because it cannot capture the real curvature of the function. The models of degree 2 and 16, however, can capture this curvature. In order to choose one model we can follow the Occam's razor: we choose the simplest one which gives the acceptable result - here, this would be the model of degree 2, as degree 16 overfits this small dataset. 

Next, we can investigate the effect of the regularization parameter $\lambda$ (function parameter `alpha`), For this, you can use the following code 

In [ ]:
polynomial_features = PolynomialFeatures(degree=1,
                                             include_bias=False)
ridge_regression = Ridge(alpha=0.001)
pipeline_ridge = Pipeline([("polynomial_features", polynomial_features),
                     ("ridge_regression", ridge_regression)])
pipeline_ridge.fit(X[:, np.newaxis], y)


For plotting you can reuse the following code

In [ ]:
#plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]), label="model of degree with regularization strength")
#plt.scatter(X, y, label='noisy samples')
#plt.plot(X_test, true_function(X_test), c='r', label='true function')
#plt.legend()

In [ ]:
#let's combine everything again
degrees = [1, 2, 16]
for degree in degrees: 
    fig, ax = plt.subplots(figsize=(8, 5),dpi=300) #just to have a nice picture
    alphas = [0, 0.001, 0.01, 1]
    colors = plt.cm.magma(np.linspace(0.35, 0.85, len(alphas))) #and of course we'll use our favourite magma 
    for alpha, color in zip(alphas,colors) :
        polynomial_features = PolynomialFeatures(degree=degree,
                                                include_bias=False)
        ridge_regression = LinearRegression() if alpha == 0 else Ridge(alpha=alpha) #chnaged this because for degree 16 a singular matrix was produced
        pipeline_ridge = Pipeline([("polynomial_features", polynomial_features),
                            ("ridge_regression", ridge_regression)])
        pipeline_ridge.fit(X[:, np.newaxis], y)

        plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]),
                color=color,
                label=f"model of degree {degree} with regularization strength {alpha}")

    plt.scatter(X, y, s=20, color=plt.cm.magma(0.15), label="noisy samples")
    plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.15), label="true function")
    plt.legend()
    plt.show()
    


Here, the same principle applies: all model with the degree of 1 perform badly, whereas the models with degrees of 2 and 16 perform quite well. The Occam's razor can be followed here with respect to the regularisation strength: restricting the magnitude in coefficients reduces the model's complexity without changing its polynomial degree. However, one should keep in mind that excessive regularisation can produce overfitting.

#### 2.16 What do you observe if you change the number of samples from the function?

In [ ]:
#let's change the number of samples
n_samples = 100 # the number of points we will sample from true_function
degrees = [1, 2, 16] # the polynomial degrees we will test

X = np.sort(np.random.rand(n_samples))
y = true_function(X) + np.random.randn(n_samples) * 0.1 # add some scaled random noise

degrees = [1, 2, 16]
for degree in degrees: 
    fig, ax = plt.subplots(figsize=(8, 5),dpi=300) #just to have a nice picture
    alphas = [0, 0.001, 0.01, 1]
    colors = plt.cm.magma(np.linspace(0.35, 0.85, len(alphas))) #and of course we'll use our favourite magma 
    for alpha, color in zip(alphas,colors) :
        polynomial_features = PolynomialFeatures(degree=degree,
                                                include_bias=False)
        ridge_regression = LinearRegression() if alpha == 0 else Ridge(alpha=alpha) #chnaged this because for degree 16 a singular matrix was produced
        pipeline_ridge = Pipeline([("polynomial_features", polynomial_features),
                            ("ridge_regression", ridge_regression)])
        pipeline_ridge.fit(X[:, np.newaxis], y)

        plt.plot(X_test, pipeline_ridge.predict(X_test[:, np.newaxis]),
                color=color,
                label=f"model of degree {degree} with regularization strength {alpha}")

    plt.scatter(X, y, s=10, color=plt.cm.magma(0.15), label="noisy samples")
    plt.plot(X_test, true_function(X_test), color=plt.cm.magma(0.25), label="true function")
    plt.legend()
    plt.show()
    



Generally, the more samples we have, the better we can fit the polynomial. Also, the polynomials of high degrees, which were overfitting quite strongly with low number of datapoints, perform significantly better. 

#### 2.17 Why do we need a test set in machine learning?

Test set is needed to estimate how good the model's performance is on unseen data. It can show whether the model generalises well or whether it has just "learnt by heart" the training set and overfits.

#### 2.18 If we need to optimize hyperparameters, do we use the test set to select the best hyperparameters? (1 point)

No, to optimise the hyperparameters we usually use the validation set. The test set is used only for the final evaluation, because otherwise the evaluation would be too optimistic.  